# SDP Pipeline Decorators Explained

This notebook explains the Spark Declarative Pipelines (SDP) constructs used across three `.py` files in the `daily_revenue_medallion_pipeline` project:

| File | Layer | Decorator |
| --- | --- | --- |
| `orders_raw.py` | Bronze | `@dp.table` |
| `orders_clean.py` | Silver | `@dp.table` + `@dp.expect_or_drop` |
| `daily_revenue.py` | Gold | `@dp.materialized_view` |

The pipeline follows a **Bronze → Silver → Gold** medallion architecture.

## `@dp.table` — Streaming Table

**Used in:** `orders_raw.py` (Bronze) and `orders_clean.py` (Silver)

Declares a **streaming table** — a table that is populated by a streaming query and auto-updated as new data arrives. The decorated function returns a Spark DataFrame (streaming or batch), and SDP manages the table's lifecycle: schema, storage, and incremental refreshes.

* In **orders_raw.py**, the function returns a `readStream` (Auto Loader), so the table is continuously appended to as new JSON files land.
* In **orders_clean.py**, the function also returns a `readStream` (from the Bronze table), so Silver rows are processed incrementally as Bronze rows arrive.

**Key difference from `@dp.materialized_view`:** a streaming table can be an *input* to downstream streaming queries; a materialized view cannot.

## `@dp.materialized_view` — Materialized View

**Used in:** `daily_revenue.py` (Gold)

Declares a **materialized view** — a table whose results are pre-computed and refreshed on a schedule or on demand (not continuously). The decorated function returns a *batch* DataFrame (`spark.read`, not `readStream`).

* **daily_revenue.py** uses `spark.read.table(...)` (batch) and aggregates the full Silver table into one row per day per currency. This is ideal for reporting — you get a complete, consistent snapshot every refresh rather than partial streaming results.

## `@dp.expect_or_drop` — Data-Quality Expectations

**Used in:** `orders_clean.py` (Silver)

A **data-quality expectation** that drops any row failing the condition. Five are stacked on the Silver table:

| Expectation | What it enforces |
| --- | --- |
| `valid_order_id` | `order_id IS NOT NULL` |
| `valid_customer_id` | `customer_id IS NOT NULL` |
| `valid_amount` | `amount IS NOT NULL` |
| `valid_currency` | `currency IS NOT NULL` |
| `valid_event_ts` | `event_ts IS NOT NULL` |

**The key design:** `try_cast` in the `select()` turns bad values into NULL, then these expectations drop the NULL rows. SDP also tracks how many rows each expectation dropped in the pipeline metrics.

Other SDP expectation variants not used here:

* `@dp.expect_or_warn` — keeps violating rows, logs a warning in metrics
* `@dp.expect` — alias for warn behavior

## Other SDP-Relevant Patterns in the Files

| Pattern | File | What it does in SDP |
| --- | --- | --- |
| `spark.readStream.format("cloudFiles")` | orders_raw.py | **Auto Loader** — SDP-native streaming file source that incrementally discovers new files |
| `spark.readStream.table()` | orders_clean.py | Streaming read from an upstream pipeline table — SDP tracks the lineage |
| `spark.read.table()` | daily_revenue.py | Batch read — materialized views always read the current full snapshot |
| `.withWatermark("event_ts", "30 minutes")` | orders_clean.py | Tells SDP/Spark how late data can arrive before state is dropped |
| `.dropDuplicatesWithinWatermark(["order_id"])` | orders_clean.py | Streaming-safe deduplication — only deduplicates within the watermark window |
| `try_cast(... AS ...)` | orders_clean.py | Safe cast that returns NULL instead of failing the query — pairs with expectations |

## Medallion Flow Summary

```
orders_raw.py          orders_clean.py              daily_revenue.py
─────────────          ──────────────                ───────────────
@dp.table              @dp.table                     @dp.materialized_view
(streaming)            (streaming)                   (batch refresh)

Auto Loader    ──▶     readStream    ──▶             read
  (Bronze)               cast + expect                  groupBy + agg
                         watermark + dedup              (Gold)
```

### Bronze (`orders_raw.py`)
* `@dp.table` declares a streaming table populated by Auto Loader
* `spark.readStream.format("cloudFiles")` incrementally reads new JSON files
* `pathGlobFilter` restricts ingestion to `orders_part_*.json` only
* No transformation — raw data is preserved as-is

### Silver (`orders_clean.py`)
* `@dp.table` declares a streaming table reading from Bronze
* `try_cast` safely converts columns to business types (NULL on failure)
* `@dp.expect_or_drop` drops rows where any cast failed (NULL check)
* `withWatermark` + `dropDuplicatesWithinWatermark` removes duplicate `order_id` values within a 30-minute late-tolerance window

### Gold (`daily_revenue.py`)
* `@dp.materialized_view` declares a batch-refreshed view (not streaming)
* `spark.read.table()` reads the full Silver snapshot
* `groupBy` currency + calendar day, then `agg` for order count, revenue, and average order value
* `event_date` cast to TIMESTAMP at midnight to match the Lab 01 gold table shape